# This notebook is used in Google Colab

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121
# !pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.5.1+cu121.html
!pip install torch-geometric

print("✅ Installation complete. NOW RESTART YOUR RUNTIME!")

In [ ]:
!pip install category_encoders

In [ ]:
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch_geometric.data import HeteroData

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from category_encoders import TargetEncoder

import GNN_create

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
nodes      = pd.read_csv("nodes.csv")
edges      = pd.read_csv("edges.csv")
labels     = pd.read_csv("labels.csv")
edge_index = pd.read_csv("edge_index.csv")

print(f"nodes      : {nodes.shape}    columns: {list(nodes.columns)}")
print(f"edges      : {edges.shape}    columns: {list(edges.columns)}")
print(f"labels     : {labels.shape}   columns: {list(labels.columns)}")
print(f"edge_index : {edge_index.shape}  columns: {list(edge_index.columns)}")
print(f"\nLabel distribution:\n{labels['Is Laundering'].value_counts()}")

In [ ]:
nodes_enc  = nodes.copy()
edges_enc  = edges.copy()

# ── Node preprocessing ───────────────────────────────────────────────────────

NODE_NUMERIC_COLS = [
    "avg_daily_txn_sent", "avg_daily_amount_sent",
    "avg_daily_txn_received", "avg_daily_amount_received",
]

# Target-encode Bank ID using account-level laundering label
te_bank = TargetEncoder(cols=["Bank ID"])
nodes_enc["Bank ID"] = te_bank.fit_transform(
    nodes_enc[["Bank ID"]], labels["Is Laundering"]
)["Bank ID"]

# Standard-scale numeric node columns (Is Laundering is binary — kept as-is)
node_scaler = StandardScaler()
nodes_enc[NODE_NUMERIC_COLS] = node_scaler.fit_transform(nodes_enc[NODE_NUMERIC_COLS])

NODE_FEATURE_COLS = ["Bank ID"] + NODE_NUMERIC_COLS + ["Is Laundering"]
print("Node feature columns:", NODE_FEATURE_COLS)

# ── Edge preprocessing ───────────────────────────────────────────────────────

EDGE_NUMERIC_COLS = ["Amount Received", "Amount Paid", "day", "hour"]

# Target-encode currency columns using transaction-level laundering label
te_recv_ccy = TargetEncoder(cols=["Receiving Currency"])
te_pay_ccy  = TargetEncoder(cols=["Payment Currency"])

edges_enc["Receiving Currency"] = te_recv_ccy.fit_transform(
    edges_enc[["Receiving Currency"]], edges_enc["Is Laundering"]
)["Receiving Currency"]

edges_enc["Payment Currency"] = te_pay_ccy.fit_transform(
    edges_enc[["Payment Currency"]], edges_enc["Is Laundering"]
)["Payment Currency"]

# Standard-scale numeric edge columns (Is Laundering is binary — kept as-is)
edge_scaler = StandardScaler()
edges_enc[EDGE_NUMERIC_COLS] = edge_scaler.fit_transform(edges_enc[EDGE_NUMERIC_COLS])

EDGE_FEATURE_COLS = [
    "Amount Received", "Receiving Currency",
    "Amount Paid",     "Payment Currency",
    "Is Laundering",   "day", "hour",
]
print("Edge feature columns:", EDGE_FEATURE_COLS)

# ── Label-encode account IDs from edge_index ─────────────────────────────────
# All unique senders + receivers → integer indices 0..N-1 that map to rows in nodes/labels

all_accounts = pd.concat([edge_index["sender"], edge_index["receiver"]]).unique()
le_acc = LabelEncoder().fit(all_accounts)

src_idx = le_acc.transform(edge_index["sender"].values)   # shape [E]
dst_idx = le_acc.transform(edge_index["receiver"].values) # shape [E]

N = len(le_acc.classes_)
print(f"\n{N:,} unique account IDs label-encoded  (nodes df has {len(nodes):,} rows)")

In [ ]:
graph = HeteroData()

# ── Node features & labels ────────────────────────────────────────────────────
graph["account"].x = torch.tensor(
    nodes_enc[NODE_FEATURE_COLS].values.astype(np.float32)
)
graph["account"].y = torch.tensor(
    labels["Is Laundering"].values.astype(np.float32)
).reshape(-1, 1)

# ── Edge connectivity & features ──────────────────────────────────────────────
graph["account", "transaction", "account"].edge_index = torch.tensor(
    np.stack([src_idx, dst_idx], axis=0), dtype=torch.long
)
graph["account", "transaction", "account"].edge_attr = torch.tensor(
    edges_enc[EDGE_FEATURE_COLS].values.astype(np.float32)
)

# ── Move graph to GPU ─────────────────────────────────────────────────────────
graph = graph.to(device)

print(f"Node type   : account")
print(f"  x shape  : {graph['account'].x.shape}")
print(f"  y shape  : {graph['account'].y.shape}")
print(f"  fraud rate: {graph['account'].y.mean().item():.4%}")
print(f"\nEdge type   : ('account', 'transaction', 'account')")
print(f"  edge_index: {graph['account', 'transaction', 'account'].edge_index.shape}")
print(f"  edge_attr : {graph['account', 'transaction', 'account'].edge_attr.shape}")
print(f"\nGraph device: {graph['account'].x.device}")


In [ ]:
RANDOM_SEED = 42

y = graph["account"].y[:, 0].cpu().numpy()
n = len(y)
indices = np.arange(n)

train_idx, test_idx = train_test_split(
    indices, test_size=0.30, random_state=RANDOM_SEED, stratify=y
)

train_mask = torch.zeros(n, dtype=torch.bool)
test_mask  = torch.zeros(n, dtype=torch.bool)
train_mask[train_idx] = True
test_mask[test_idx]   = True

graph["account"].train_mask = train_mask.to(device)
graph["account"].test_mask  = test_mask.to(device)

print(f"Total accounts : {n:,}")
print(f"Train          : {len(train_idx):,}  (fraud: {y[train_idx].sum():.0f})")
print(f"Test           : {len(test_idx):,}  (fraud: {y[test_idx].sum():.0f})")


In [ ]:
model = GNN_create.HMPNN_ct_3Layer(graph, node_type="account").to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model        : HMPNN_ct_3Layer")
print(f"Node type    : account")
print(f"Parameters   : {total_params:,}")
print(f"Model device : {next(model.parameters()).device}")


In [ ]:
LEARNING_RATE       = 1e-3
WEIGHT_DECAY        = 1e-5
MIN_EPOCHS          = 200
MAX_EPOCHS          = 2_000
PRINT_FREQ          = 50
EARLY_STOP_PATIENCE = 100

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Weighted BCE: pos_weight = num_negatives / num_positives to handle class imbalance
y_train = graph["account"].y[graph["account"].train_mask, 0]
n_pos   = y_train.sum().item()
n_neg   = (y_train == 0).sum().item()
pos_weight = torch.tensor([n_neg / n_pos]).to(device)  # must be on same device as model
loss_fn = nn.BCELoss(weight=pos_weight)

print(f"pos_weight = {pos_weight.item():.1f}  (train positives={n_pos:.0f}, negatives={n_neg:.0f})")


In [ ]:
x_dict          = graph.x_dict
edge_index_dict = graph.edge_index_dict
edge_attr_dict  = graph.edge_attr_dict

history = {"train_loss": [], "test_loss": []}
best_test_loss   = float("inf")
patience_counter = 0
best_state       = None

for epoch in range(1, MAX_EPOCHS + 1):

    # ── Training step ──────────────────────────────────────────────────────
    model.train()
    optimizer.zero_grad()

    pred = model(x_dict, edge_index_dict, edge_attr_dict)  # Tensor [N, 1]

    train_mask = graph["account"].train_mask
    train_loss = loss_fn(pred[train_mask], graph["account"].y[train_mask])
    train_loss.backward()
    optimizer.step()

    # ── Test loss (no gradient) ────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        pred_eval = model(x_dict, edge_index_dict, edge_attr_dict)
        test_mask = graph["account"].test_mask
        test_loss = loss_fn(pred_eval[test_mask], graph["account"].y[test_mask])

    history["train_loss"].append(train_loss.item())
    history["test_loss"].append(test_loss.item())

    if epoch % PRINT_FREQ == 0 or epoch == 1:
        print(f"Epoch {epoch:5d}  train_loss={train_loss.item():.4f}  "
              f"test_loss={test_loss.item():.4f}")

    # ── Early stopping ────────────────────────────────────────────────────
    if test_loss.item() < best_test_loss:
        best_test_loss   = test_loss.item()
        best_state       = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch >= MIN_EPOCHS and patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}  (best test loss = {best_test_loss:.4f})")
        break

model.load_state_dict(best_state)
print("Training complete — best weights restored.")

In [ ]:
# ── 10a. Training curve ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history["train_loss"], label="Train loss", color="steelblue")
ax.plot(history["test_loss"],  label="Test loss",  color="coral")
ax.set_xlabel("Epoch")
ax.set_ylabel("Weighted BCE loss (sum over entity types)")
ax.set_title("Training curve")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC curve ─────────────────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    final_pred = model(x_dict, edge_index_dict, edge_attr_dict)

y_true = graph["account"].y[test_mask, 0].cpu().numpy()
y_prob = final_pred[test_mask, 0].cpu().numpy()

fpr, tpr, thresholds = roc_curve(y_true, y_prob)
auc = roc_auc_score(y_true, y_prob)

# Youden-optimal threshold: maximises (sensitivity + specificity - 1)
j_idx       = np.argmax(tpr - fpr)
opt_thresh  = thresholds[j_idx]

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color="steelblue", linewidth=2, label=f"AUC = {auc:.4f}")
ax.scatter(fpr[j_idx], tpr[j_idx], color="red", zorder=5,
           label=f"Optimal threshold = {opt_thresh:.4f}")
ax.plot([0, 1], [0, 1], "--", color="grey")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Account-level AML detection (test split)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Test AUC: {auc:.4f}  |  Optimal threshold: {opt_thresh:.4f}")


In [ ]:
# ── Classification report at Youden-optimal threshold ─────────────────────────
y_pred = (y_prob >= opt_thresh).astype(int)

print(f"Threshold : {opt_thresh:.4f}")
print(f"AUC       : {auc:.4f}\n")
print(classification_report(y_true, y_pred, target_names=["Legit", "Fraud"], digits=4))

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix (rows=actual, cols=predicted):")
print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
print(f"  FN={cm[1,0]}  TP={cm[1,1]}")

In [ ]:
import os, pickle

save_dir = "model_artifacts"
os.makedirs(save_dir, exist_ok=True)

# Model weights
torch.save(model.state_dict(), f"{save_dir}/aml_hmpnn_ct_3layer.pt")

# Preprocessing artifacts needed for inference on new data
artifacts = {
    "te_bank":            te_bank,
    "te_recv_ccy":        te_recv_ccy,
    "te_pay_ccy":         te_pay_ccy,
    "le_acc":             le_acc,
    "node_scaler":        node_scaler,
    "edge_scaler":        edge_scaler,
    "node_feature_cols":  NODE_FEATURE_COLS,
    "edge_feature_cols":  EDGE_FEATURE_COLS,
    "opt_thresh":         opt_thresh,
}
with open(f"{save_dir}/preprocessing_artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print(f"Saved to '{save_dir}/':")
print(f"  aml_hmpnn_ct_3layer.pt")
print(f"  preprocessing_artifacts.pkl")